# aerial-vg: extra seeds (14, 15) for baseline and d4_remap

Same notebook, code dataset and environment as Version 5 (seed 13); only the seed and run names differ.
Phase 1: baseline_s14 (GPU0) ‖ d4_remap_s14 (GPU1). Phase 2: baseline_s15 ‖ d4_remap_s15. ~7.3 h.
Also records the data MD5 and the image overlap between the released splits.


In [1]:
import time; NB_START = time.time()   # used to keep the whole run inside Kaggle's 12 h limit
REPO_URL = 'https://github.com/HonourGok/aerial-vg.git'   # used only if the code is not attached as a Kaggle dataset
import os, subprocess, glob, shutil
DATA_ROOT = '/tmp/DIOR_RSVG'   # /tmp is not saved as notebook output, so the 5 GB stay out of the 20 GB output limit

def find_dir(root, ext):
    best = None
    for d, _, files in os.walk(root):
        n = sum(f.lower().endswith(ext) for f in files)
        if n and (best is None or n > best[1]):
            best = (d, n)
    return best

if not os.path.isdir(DATA_ROOT + '/Annotations'):
    # 1) a manually uploaded Kaggle dataset, if one was added as input
    up = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/Annotations', recursive=True)]
    if up:
        DATA_ROOT = up[0]
    else:
        # 2) download from the authors' Google Drive folder (Annotations.zip 7 MB, JPEGImages.zip 5 GB)
        subprocess.run('pip -q install gdown', shell=True, check=True)
        os.makedirs('/tmp/rsvg_dl', exist_ok=True)
        subprocess.run('gdown --folder https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_ -O /tmp/rsvg_dl',
                       shell=True, check=True)
        os.makedirs(DATA_ROOT, exist_ok=True)
        for z in glob.glob('/tmp/rsvg_dl/**/*.zip', recursive=True):
            subprocess.run(f'md5sum "{z}" | tee -a /tmp/data_md5.txt', shell=True)   # data version
            subprocess.run(f'unzip -q -o "{z}" -d /tmp/rsvg_unz', shell=True, check=True)
            os.remove(z)
        for sub, ext in (('Annotations', '.xml'), ('JPEGImages', '.jpg')):
            d, n = find_dir('/tmp/rsvg_unz', ext)
            print(sub, '<-', d, n, 'files')
            shutil.move(d, f'{DATA_ROOT}/{sub}')

print('DATA_ROOT =', DATA_ROOT, '| xml:', len(os.listdir(DATA_ROOT + '/Annotations')),
      '| jpg:', len(os.listdir(DATA_ROOT + '/JPEGImages')))
# If gdown fails with a quota / "too many users" error: download the two zips manually and upload them as a
# Kaggle dataset (Datasets -> New Dataset), add it as input, and re-run this cell.

Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1Cz4wMifO32jLmQdxtsGyJuNjSbOxnbG_
To: /tmp/rsvg_dl/Annotations.zip
100%|██████████| 7.77M/7.77M [00:00<00:00, 103MB/s]


Processing file 1Cz4wMifO32jLmQdxtsGyJuNjSbOxnbG_ Annotations.zip
Processing file 1pdUl2MHXqo70dbw7BdUhkW25boveDJfo JPEGImages.zip
Processing file 14rEjQ9Mp92rkw6HoDD5MbVTMLjA0Zx1G test.txt
Processing file 12W0HZdS7ri1Xf5dyB90JyPL4dPUv7ogA train.txt
Processing file 1-9-LcZtsK663xdmtW1DtYjKztEMaroNz val.txt


Downloading...
From (original): https://drive.google.com/uc?id=1pdUl2MHXqo70dbw7BdUhkW25boveDJfo
From (redirected): https://drive.google.com/uc?id=1pdUl2MHXqo70dbw7BdUhkW25boveDJfo&confirm=t&uuid=6926314f-55a7-4ea8-9c6a-003f334df9be
To: /tmp/rsvg_dl/JPEGImages.zip
100%|██████████| 5.39G/5.39G [00:47<00:00, 114MB/s] 
Downloading...
From: https://drive.google.com/uc?id=14rEjQ9Mp92rkw6HoDD5MbVTMLjA0Zx1G
To: /tmp/rsvg_dl/test.txt
100%|██████████| 42.9k/42.9k [00:00<00:00, 64.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=12W0HZdS7ri1Xf5dyB90JyPL4dPUv7ogA
To: /tmp/rsvg_dl/train.txt
100%|██████████| 154k/154k [00:00<00:00, 95.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-9-LcZtsK663xdmtW1DtYjKztEMaroNz
To: /tmp/rsvg_dl/val.txt
100%|██████████| 21.9k/21.9k [00:00<00:00, 39.7MB/s]
Download completed


c4bda6c8a36074870d048cf4228a9d71  /tmp/rsvg_dl/JPEGImages.zip
46e5a701ec05236df23a232e39b99599  /tmp/rsvg_dl/Annotations.zip
Annotations <- /tmp/rsvg_unz/Annotations 17402 files
JPEGImages <- /tmp/rsvg_unz/JPEGImages 17402 files
DATA_ROOT = /tmp/DIOR_RSVG | xml: 17402 | jpg: 17402


In [2]:
%cd /kaggle/working
# code: from an attached Kaggle dataset containing aerial-vg/ (works while the GitHub repo is private), else git clone
code_src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/aerial-vg/train.py', recursive=True)]
!rm -rf aerial-vg
if code_src:
    shutil.copytree(code_src[0], '/kaggle/working/aerial-vg')
    print('code from', code_src[0])
else:
    subprocess.run(f'git clone -q {REPO_URL} aerial-vg', shell=True, check=True)
%cd /kaggle/working/aerial-vg
!(git rev-parse HEAD 2>/dev/null || md5sum train.py datasets/geo_aug.py) | tee results_commit.txt
!pip -q install -r requirements.txt
!mkdir -p checkpoints && wget -q -nc -O checkpoints/detr-r50-e632da11.pth https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch,torchvision,transformers;print(torch.__version__,torchvision.__version__,transformers.__version__)" 

/kaggle/working
code from /kaggle/input/datasets/honourgok/aerial-vg-code/aerial-vg
/kaggle/working/aerial-vg
ff198d0091861a2eb5b1b08175309684  train.py
849c9ccb4be023a5cab470a35b19fb37  datasets/geo_aug.py
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
2.11.0+cu130 0.26.0+cu130 5.20.2


In [3]:
# one-off: resize images to 640 px (the model input size) so the data loader does not decode 800 px JPEGs every epoch
if not os.path.isdir('/tmp/DIOR_RSVG_640/JPEGImages'):
    !python tools/preresize.py --src $DATA_ROOT --dst /tmp/DIOR_RSVG_640 --size 640
DATA_ROOT = '/tmp/DIOR_RSVG_640'
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['DETR'] = '/kaggle/working/aerial-vg/checkpoints/detr-r50-e632da11.pth'
# compute budget (identical for all conditions)
# measured on T4: 17.5 train samples/s -> ~26 min per epoch on the full train split, ~5 min for the full test set
SMOKE = False   # True = 2-minute runs on tiny data, to check the whole notebook end to end in Save & Run All
if SMOKE:
    os.environ.update(EPOCHS='1', LR_DROP='1', FRACTION='0.005', VAL_MAX='40', TEST_LIMIT='200', LAT='20', MAX_HOURS='1')
else:
    os.environ.update(EPOCHS='7', LR_DROP='5', FRACTION='1.0', VAL_MAX='500', TEST_LIMIT='0', LAT='200', MAX_HOURS='5.0')
os.environ.update(BS='8', IMSIZE='640', SEED='13')
print('SMOKE =', SMOKE, {k: os.environ[k] for k in ('EPOCHS', 'LR_DROP', 'FRACTION', 'VAL_MAX', 'TEST_LIMIT', 'MAX_HOURS')})
!python tools/dataset_stats.py --data_root $DATA_ROOT --split_root splits --out results/dataset_stats.json

resized 17402 images, rewrote 17402 xml files -> /tmp/DIOR_RSVG_640
SMOKE = False {'EPOCHS': '7', 'LR_DROP': '5', 'FRACTION': '1.0', 'VAL_MAX': '500', 'TEST_LIMIT': '0', 'MAX_HOURS': '5.0'}
train {'n_expressions': 26991, 'n_images': 14730, 'pct_with_direction_word': 62.16887110518321, 'direction_word_counts': {'left': 6890, 'right': 7137, 'top': 2076, 'bottom': 2288, 'upper': 3854, 'lower': 3872, 'leftmost': 1, 'above': 548, 'below': 618}, 'mean_words': 7.439257530287874}
val {'n_expressions': 3829, 'n_images': 3465, 'pct_with_direction_word': 63.59362757900235, 'direction_word_counts': {'left': 1027, 'right': 1007, 'top': 315, 'bottom': 346, 'upper': 564, 'lower': 582, 'above': 79, 'below': 81}, 'mean_words': 7.523896578741185}
test {'n_expressions': 7500, 'n_images': 6102, 'pct_with_direction_word': 62.25333333333333, 'direction_word_counts': {'left': 1938, 'right': 1995, 'top': 575, 'bottom': 632, 'upper': 1097, 'lower': 1059, 'above': 157, 'below': 157}, 'mean_words': 7.46613333333

In [4]:
# FIX (diagnosed 2026-11-10): the post-LN fusion transformer collapsed to one constant box for every input (loss stuck ~1.85, all 4 runs).
# Pre-LN fusion + 500-iter linear LR warmup + grad-clip 0.1 fixes it (1 epoch full data: val Pr@0.5 26% vs ~2% before). Applied identically to all conditions.
import re; p='/kaggle/working/aerial-vg/'
s=open(p+'models/vl_transformer.py').read(); i=s.index('def build_vl_transformer'); t=s[i:]; assert t.count('normalize_before=False')==1; open(p+'models/vl_transformer.py','w').write(s[:i]+t.replace('normalize_before=False',"normalize_before=bool(getattr(args, 'vl_prenorm', 0))"))
s=open(p+'train.py').read(); m=re.search(r"\n( *)parser\.add_argument\('--amp'", s); ind=m.group(1); s=s[:m.start()]+"\n"+ind+"parser.add_argument('--vl_prenorm', default=1, type=int, help='pre-LN fusion transformer (post-LN collapses to a constant box)')\n"+ind+"parser.add_argument('--warmup_iters', default=0, type=int, help='linear LR warmup iterations in epoch 0')"+s[m.start():]; open(p+'train.py','w').write(s)
s=open(p+'engine.py').read(); m=re.search(r"\n( *)img_data, text_data, target = batch\n", s); ind=m.group(1); add=ind+"it = getattr(args, '_it', 0); args._it = it + 1\n"+ind+"if epoch == 0 and it < getattr(args, 'warmup_iters', 0):\n"+ind+"    for pg in optimizer.param_groups:\n"+ind+"        pg['lr'] = pg.setdefault('initial_lr', pg['lr']) * (it + 1) / args.warmup_iters\n"; open(p+'engine.py','w').write(s[:m.end()]+add+s[m.end():])
s=open(p+'scripts/run_one.sh').read(); assert s.count('python train.py')==1; open(p+'scripts/run_one.sh','w').write(s.replace('python train.py','python train.py --vl_prenorm 1 --warmup_iters 500 --clip_max_norm 0.1'))
!cd /kaggle/working/aerial-vg && python -m py_compile train.py engine.py models/vl_transformer.py && grep -n "vl_prenorm\|warmup_iters\|clip_max_norm 0.1" models/vl_transformer.py train.py engine.py scripts/run_one.sh
# Time budget: remaining GPU quota is ~10 h, so the whole commit must finish well inside it (and inside Kaggle's 12 h): target <= 9.3 h total.
RUN_BUDGET_H = 9.3
if not SMOKE: os.environ['MAX_HOURS'] = '4.2'
print('RUN_BUDGET_H =', RUN_BUDGET_H, '| phase-1 MAX_HOURS =', os.environ['MAX_HOURS'])

models/vl_transformer.py:139:        normalize_before=bool(getattr(args, 'vl_prenorm', 0)),
train.py:127:    parser.add_argument('--vl_prenorm', default=1, type=int, help='pre-LN fusion transformer (post-LN collapses to a constant box)')
train.py:128:    parser.add_argument('--warmup_iters', default=0, type=int, help='linear LR warmup iterations in epoch 0')
engine.py:31:        if epoch == 0 and it < getattr(args, 'warmup_iters', 0):
engine.py:33:                pg['lr'] = pg.setdefault('initial_lr', pg['lr']) * (it + 1) / args.warmup_iters
scripts/run_one.sh:19:CUDA_VISIBLE_DEVICES=$GPU python train.py --vl_prenorm 1 --warmup_iters 500 --clip_max_norm 0.1 \
RUN_BUDGET_H = 9.3 | phase-1 MAX_HOURS = 4.2


In [5]:
# Data checks (no training): data version and image overlap between the released splits.
import json, xml.etree.ElementTree as ET
if os.path.exists('/tmp/data_md5.txt'): print(open('/tmp/data_md5.txt').read())
objs = []   # running object index in sorted XML order, same as datasets/dior_rsvg.py
for f in sorted(f for f in os.listdir(os.environ['DATA_ROOT'] + '/Annotations') if f.endswith('.xml')):
    r = ET.parse(os.path.join(os.environ['DATA_ROOT'], 'Annotations', f)).getroot()
    objs += [r.find('filename').text] * len(r.findall('object'))
imgs = {s: {objs[int(l.split()[0])] for l in open(f'splits/{s}.txt') if l.strip()} for s in ('train', 'val', 'test')}
ov = {'objects': len(objs), **{s: len(v) for s, v in imgs.items()},
      'train&val': len(imgs['train'] & imgs['val']), 'train&test': len(imgs['train'] & imgs['test']), 'val&test': len(imgs['val'] & imgs['test'])}
print(ov); os.makedirs('results', exist_ok=True); json.dump(ov, open('results/split_image_overlap.json', 'w'), indent=1)


c4bda6c8a36074870d048cf4228a9d71  /tmp/rsvg_dl/JPEGImages.zip
46e5a701ec05236df23a232e39b99599  /tmp/rsvg_dl/Annotations.zip

{'objects': 38320, 'train': 14730, 'val': 3465, 'test': 6102, 'train&val': 2449, 'train&test': 4206, 'val&test': 1115}


In [6]:
# Speed check (~3 min, interactive sessions only; skipped in Save & Run All)
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') != 'Batch':
    !CUDA_VISIBLE_DEVICES=0 python tools/benchmark.py --dataset dior_rsvg --data_root $DATA_ROOT --split_root splits \
        --detr_model $DETR --imsize 640 --max_query_len 40 --batch_size 8 --amp --num_workers 4 --iters 40 \
        --aug_crop --aug_scale --aug_translate --geo_aug d4_remap --train_fraction 0.5 --val_max 500 2>&1 | grep -v Warning

In [7]:
# Phase 1: seed 14, baseline (GPU0) || proposed (GPU1)
# Jobs are started with subprocess.Popen and waited on directly; a failing job stops the cell within ~20 s.
import sys, importlib
sys.path.insert(0, 'scripts'); import launch; importlib.reload(launch)
# strict (stop on first failure) only in SMOKE; in the long run a failure of one job must not kill the other
os.environ['SEED'] = '14'
launch.run_parallel([('baseline_s14', 'hflip', 0), ('d4_remap_s14', 'd4_remap', 1)], strict=SMOKE)

started baseline_s14 (geo_aug=hflip, GPU 0), pid 124
started d4_remap_s14 (geo_aug=d4_remap, GPU 1), pid 125
[0.01 h] baseline_s14: number of params: 149529092
[0.01 h] d4_remap_s14: number of params: 149529092
[0.17 h] baseline_s14: Epoch: [0]  [1130/3373]  eta: 0:19:27  lr: 0.000100  loss: 1.5325 (1.6396)  loss_bbox: 0.5194 (0.5874)  loss_giou: 1.0131 (1.0522)  time: 0.5187  data: 0.0045  
[0.17 h] d4_remap_s14: Epoch: [0]  [1130/3373]  eta: 0:19:26  lr: 0.000100  loss: 1.6161 (1.6371)  loss_bbox: 0.5706 (0.5821)  loss_giou: 1.0455 (1.0550)  time: 0.5093  data: 0.0037  
[0.34 h] baseline_s14: Epoch: [0]  [2290/3373]  eta: 0:09:21  lr: 0.000100  loss: 1.2770 (1.5226)  loss_bbox: 0.3543 (0.5074)  loss_giou: 0.9228 (1.0152)  time: 0.5108  data: 0.0040  
[0.34 h] d4_remap_s14: Epoch: [0]  [2290/3373]  eta: 0:09:21  lr: 0.000100  loss: 1.3523 (1.5212)  loss_bbox: 0.3448 (0.5027)  loss_giou: 1.0075 (1.0185)  time: 0.5147  data: 0.0045  
[0.51 h] baseline_s14: Epoch: [1]  [  20/3373]  eta: 

{'d4_remap_s14': 0, 'baseline_s14': 0}

In [8]:
# Phase 2: seed 15, baseline (GPU0) || proposed (GPU1)
# Time guard: everything must finish before Kaggle's 12 h limit, otherwise no output is saved.
# Runs stop after the epoch that crosses MAX_HOURS (one epoch ~0.5 h) and are then evaluated (~0.2 h).
elapsed = (time.time() - NB_START) / 3600
if not SMOKE:
    os.environ['MAX_HOURS'] = '%.2f' % max(0.5, min(5.0, RUN_BUDGET_H - elapsed - 0.5 - 0.3))
print('elapsed %.2f h, phase-2 MAX_HOURS = %s' % (elapsed, os.environ['MAX_HOURS']))
os.environ['SEED'] = '15'
launch.run_parallel([('baseline_s15', 'hflip', 0), ('d4_remap_s15', 'd4_remap', 1)], strict=SMOKE)

elapsed 3.67 h, phase-2 MAX_HOURS = 4.83
started baseline_s15 (geo_aug=hflip, GPU 0), pid 1167
started d4_remap_s15 (geo_aug=d4_remap, GPU 1), pid 1168
[0.01 h] baseline_s15: Epoch: [0]  [   0/3373]  eta: 3:28:10  lr: 0.000000  loss: 1.7042 (1.7042)  loss_bbox: 0.7961 (0.7961)  loss_giou: 0.9081 (0.9081)  time: 3.7031  data: 1.3810  
[0.01 h] d4_remap_s15: Epoch: [0]  [   0/3373]  eta: 3:24:10  lr: 0.000000  loss: 1.7697 (1.7697)  loss_bbox: 0.8146 (0.8146)  loss_giou: 0.9551 (0.9551)  time: 3.6319  data: 0.8237  
[0.17 h] baseline_s15: Epoch: [0]  [1150/3373]  eta: 0:19:20  lr: 0.000100  loss: 1.2704 (1.6920)  loss_bbox: 0.4657 (0.6165)  loss_giou: 0.8047 (1.0755)  time: 0.5040  data: 0.0056  
[0.17 h] d4_remap_s15: Epoch: [0]  [1160/3373]  eta: 0:19:08  lr: 0.000100  loss: 1.4891 (1.6569)  loss_bbox: 0.4786 (0.6006)  loss_giou: 1.0105 (1.0563)  time: 0.5049  data: 0.0031  
[0.34 h] baseline_s15: Epoch: [0]  [2310/3373]  eta: 0:09:12  lr: 0.000100  loss: 1.4888 (1.5558)  loss_bbox: 0.

{'d4_remap_s15': 0, 'baseline_s15': 0}

In [9]:
# Collect: comparison over the runs that produced metrics, figure, training logs, zip
ALL = ['baseline_s14', 'd4_remap_s14', 'baseline_s15', 'd4_remap_s15']
runs = [r for r in ALL if os.path.exists(f'results/{r}/metrics.json')]
print('runs with results:', runs)
if False:   # comparison over seeds is done offline with tools/seed_summary.py
    subprocess.run(['python', 'compare.py', '--runs'] + [f'{r}=results/{r}' for r in runs] +
                   ['--ref', 'baseline', '--out', 'results/comparison'], check=False)
if False:
    subprocess.run(['python', 'tools/viz_examples.py', '--n', '8', '--a', 'results/baseline', '--b', 'results/d4_remap',
                    '--data_root', DATA_ROOT, '--out', 'results/figures/baseline_vs_d4remap.png'], check=False)
for r in ALL:
    if os.path.exists(f'outputs/{r}/log.txt'):
        os.makedirs(f'results/{r}', exist_ok=True)
        shutil.copy(f'outputs/{r}/log.txt', f'results/{r}/train_log.txt')
os.makedirs('results/logs', exist_ok=True)
for f in glob.glob('logs/*.log'):
    shutil.copy(f, 'results/logs/')
shutil.copy('results_commit.txt', 'results/')
print('total hours: %.2f' % ((time.time() - NB_START) / 3600))
!cd /kaggle/working && rm -f results.zip && zip -qr results.zip aerial-vg/results && ls -lh results.zip
!du -sh /kaggle/working/aerial-vg/outputs

runs with results: ['baseline_s14', 'd4_remap_s14', 'baseline_s15', 'd4_remap_s15']
total hours: 7.24
-rw-r--r-- 1 root root 1.3M Nov 16 12:23 results.zip
9.0G	/kaggle/working/aerial-vg/outputs
